# Vera Verifier — LLM-AggreFact Benchmark Evaluation

This notebook evaluates Vera's **ModernBERT-large-zeroshot-v2.0** verifier against the [LLM-AggreFact benchmark](https://huggingface.co/datasets/lytang/LLM-AggreFact) (Tang et al., EMNLP 2024 / MiniCheck).

The benchmark aggregates **11 human-annotated datasets** for factual consistency evaluation across closed-book and grounded generation settings.

- **Labels**: 1 = supported, 0 = unsupported
- **Primary Metric**: Balanced Accuracy (BAcc)
- **Model**: `MoritzLaurer/ModernBERT-large-zeroshot-v2.0` (395M params)

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = 'drive/MyDrive/vera/evaluations/llmaggrefact-modernbert'

Mounted at /content/drive


In [2]:
!pip install -q datasets transformers torch scikit-learn tqdm pandas tabulate huggingface-hub

In [ ]:
# Required because LLM-AggreFact is a gated dataset.

HF_TOKEN = None  # <-- paste your token here


from huggingface_hub import login as hf_login

if HF_TOKEN:
    hf_login(token=HF_TOKEN)
    print("✅ Authenticated with HuggingFace.")
else:
    print("⚠️  No HF token set — will fail on gated datasets. Paste your token above.")

✅ Authenticated with HuggingFace.


## 2. Load Benchmark

In [4]:
from datasets import load_dataset

SPLIT = "test"  # Change to "dev" for the development set

print(f"Loading LLM-AggreFact benchmark ({SPLIT} split)...")
benchmark = load_dataset("lytang/LLM-AggreFact", split=SPLIT, token=HF_TOKEN)
print(f"Total samples: {len(benchmark)}")

import pandas as pd
df_bench = benchmark.to_pandas()
print("\nSamples per dataset:")
print(df_bench['dataset'].value_counts().to_string())

Loading LLM-AggreFact benchmark (test split)...


README.md:   0%|          | 0.00/4.93k [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/35.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/30420 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/29320 [00:00<?, ? examples/s]

Total samples: 29320

Samples per dataset:
dataset
RAGTruth           16371
ExpertQA            3702
Lfqa                1911
Reveal              1710
FactCheck-GPT       1566
ClaimVerify         1088
TofuEval-MeetB       772
TofuEval-MediaS      726
AggreFact-CNN        558
AggreFact-XSum       558
Wice                 358


## 3. Load Model

In [5]:
from transformers import pipeline

MODEL_ID = "MoritzLaurer/ModernBERT-large-zeroshot-v2.0"
CANDIDATE_LABELS = ["entailment", "not_entailment"]
HYPOTHESIS_TEMPLATE = "This claim is {}."
THRESHOLD = 0.5

print(f"Loading model: {MODEL_ID}")
classifier = pipeline(
    "zero-shot-classification",
    model=MODEL_ID,
    device=0,  # GPU
)
print("✅ Model loaded on GPU!")

Loading model: MoritzLaurer/ModernBERT-large-zeroshot-v2.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/792M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

✅ Model loaded on GPU!


## 4. Inference Helpers

In [6]:
from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


def predict_single(doc: str, claim: str) -> dict:
    """
    Run the Vera verifier on a single (doc, claim) pair.
    Uses the same format as deployment/verifier/server.py.
    """
    sequence = f"Evidence: {doc}\n\nClaim: {claim}"

    result = classifier(
        sequence,
        CANDIDATE_LABELS,
        hypothesis_template=HYPOTHESIS_TEMPLATE,
    )

    scores = dict(zip(result["labels"], result["scores"]))
    ent_score = scores.get("entailment", 0.0)

    pred_label = 1 if ent_score > THRESHOLD else 0
    verdict = "SUPPORTED" if pred_label == 1 else "REFUTED"

    return {
        "pred_label": pred_label,
        "entailment_score": ent_score,
        "verdict": verdict,
    }


def compute_metrics(y_true, y_pred):
    """Compute evaluation metrics."""
    bacc = balanced_accuracy_score(y_true, y_pred)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "BAcc": round(bacc * 100, 1),
        "Accuracy": round(acc * 100, 1),
        "Precision": round(prec * 100, 1),
        "Recall": round(rec * 100, 1),
        "F1": round(f1 * 100, 1),
        "TP": int(tp), "TN": int(tn),
        "FP": int(fp), "FN": int(fn),
        "N": len(y_true),
        "Pos%": round(sum(y_true) / len(y_true) * 100, 1),
    }

## 5. Run Evaluation

In [7]:
import time
import json
import os
from tqdm.notebook import tqdm
from datetime import datetime

ALL_DATASETS = [
    "AggreFact-CNN", "AggreFact-XSum",
    "TofuEval-MediaS", "TofuEval-MeetB",
    "Wice", "Reveal", "ClaimVerify",
    "FactCheck-GPT", "ExpertQA", "Lfqa", "RAGTruth",
]

TARGET_DATASETS = ALL_DATASETS  # Change to e.g. ["Wice", "FactCheck-GPT"] for a quick test
MAX_SAMPLES = None              # Set to e.g. 50 for a quick smoke test

all_results = {}
all_predictions = []
overall_y_true = []
overall_y_pred = []
total_time = 0

for ds_name in TARGET_DATASETS:
    ds_samples = df_bench[df_bench['dataset'] == ds_name]

    if len(ds_samples) == 0:
        print(f"⚠ No samples found for '{ds_name}', skipping.")
        continue

    if MAX_SAMPLES and len(ds_samples) > MAX_SAMPLES:
        ds_samples = ds_samples.head(MAX_SAMPLES)

    print(f"\n{'='*60}")
    print(f"Evaluating: {ds_name} ({len(ds_samples)} samples)")
    print(f"{'='*60}")

    y_true, y_pred = [], []
    start_time = time.time()

    for _, row in tqdm(ds_samples.iterrows(), total=len(ds_samples), desc=ds_name):
        result = predict_single(row['doc'], row['claim'])

        y_true.append(row['label'])
        y_pred.append(result['pred_label'])

        all_predictions.append({
            "dataset": ds_name,
            "claim": row['claim'][:200],
            "doc_len": len(row['doc']),
            "gold_label": int(row['label']),
            "pred_label": result['pred_label'],
            "entailment_score": round(result['entailment_score'], 4),
            "verdict": result['verdict'],
            "correct": int(row['label']) == result['pred_label'],
        })

    elapsed = time.time() - start_time
    total_time += elapsed

    metrics = compute_metrics(y_true, y_pred)
    metrics["Time (s)"] = round(elapsed, 1)
    metrics["Throughput (ex/s)"] = round(len(ds_samples) / elapsed, 1)
    all_results[ds_name] = metrics

    overall_y_true.extend(y_true)
    overall_y_pred.extend(y_pred)

    print(f"  BAcc: {metrics['BAcc']}%  |  Acc: {metrics['Accuracy']}%  |  "
          f"F1: {metrics['F1']}%  |  N: {metrics['N']}  |  Time: {metrics['Time (s)']}s")

if overall_y_true:
    overall_metrics = compute_metrics(overall_y_true, overall_y_pred)
    overall_metrics["Time (s)"] = round(total_time, 1)
    overall_metrics["Throughput (ex/s)"] = round(len(overall_y_true) / total_time, 1)
    all_results["OVERALL"] = overall_metrics

print("\n✅ Evaluation complete!")


Evaluating: AggreFact-CNN (558 samples)


AggreFact-CNN:   0%|          | 0/558 [00:00<?, ?it/s]

W0222 21:14:40.274000 31676 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  BAcc: 55.1%  |  Acc: 64.0%  |  F1: 76.8%  |  N: 558  |  Time: 52.0s

Evaluating: AggreFact-XSum (558 samples)


AggreFact-XSum:   0%|          | 0/558 [00:00<?, ?it/s]

...

  BAcc: 51.5%  |  Acc: 65.7%  |  F1: 77.8%  |  N: 3702  |  Time: 283.4s

Evaluating: Lfqa (1911 samples)


Lfqa:   0%|          | 0/1911 [00:00<?, ?it/s]

  BAcc: 56.8%  |  Acc: 59.9%  |  F1: 68.5%  |  N: 1911  |  Time: 144.0s

Evaluating: RAGTruth (16371 samples)


RAGTruth:   0%|          | 0/16371 [00:00<?, ?it/s]

  BAcc: 54.4%  |  Acc: 53.6%  |  F1: 68.0%  |  N: 16371  |  Time: 1244.4s

✅ Evaluation complete!


## 6. Results Summary

In [8]:
results_df = pd.DataFrame(all_results).T
display_cols = ["BAcc", "Accuracy", "Precision", "Recall", "F1", "N", "Pos%", "Time (s)"]

print("=" * 80)
print("RESULTS — Vera ModernBERT on LLM-AggreFact")
print(f"Split: {SPLIT}  |  Threshold: {THRESHOLD}  |  Model: {MODEL_ID}")
print("=" * 80)
results_df[display_cols]

RESULTS — Vera ModernBERT on LLM-AggreFact
Split: test  |  Threshold: 0.5  |  Model: MoritzLaurer/ModernBERT-large-zeroshot-v2.0


,BAcc,Accuracy,Precision,Recall,F1,N,Pos%,Time (s)
AggreFact-CNN,55.1,64.0,91.2,66.3,76.8,558.0,89.8,52.0
AggreFact-XSum,52.9,53.0,53.6,59.6,56.5,558.0,51.1,42.2
TofuEval-MediaS,48.7,40.9,74.9,33.9,46.7,726.0,76.3,56.6
TofuEval-MeetB,50.2,42.6,80.8,37.8,51.5,772.0,80.6,60.3
Wice,48.3,50.6,29.4,42.3,34.7,358.0,31.0,30.3
Reveal,63.5,52.1,30.9,85.0,45.4,1710.0,23.4,123.2
ClaimVerify,50.3,45.1,72.9,38.8,50.6,1088.0,72.5,94.0
FactCheck-GPT,57.6,42.3,27.7,87.0,42.0,1566.0,24.0,112.9
ExpertQA,51.5,65.7,80.9,75.0,77.8,3702.0,80.3,283.4
Lfqa,56.8,59.9,63.5,74.4,68.5,1911.0,58.7,144.0


## 7. Comparison with Published Baselines

In [9]:
baselines = {
    "AlignScore (355M)": {
        "AggreFact-CNN": 52.4, "AggreFact-XSum": 71.4,
        "TofuEval-MediaS": 69.2, "TofuEval-MeetB": 72.6,
        "Wice": 66.0, "Reveal": 85.3, "ClaimVerify": 69.6,
        "FactCheck-GPT": 74.3, "ExpertQA": 58.3, "Lfqa": 84.5,
    },
    "MiniCheck-FT5 (770M)": {
        "AggreFact-CNN": 69.9, "AggreFact-XSum": 74.3,
        "TofuEval-MediaS": 73.6, "TofuEval-MeetB": 77.3,
        "Wice": 72.2, "Reveal": 86.2, "ClaimVerify": 74.6,
        "FactCheck-GPT": 74.7, "ExpertQA": 59.0, "Lfqa": 85.2,
    },
    "GPT-4 (~1.8T)": {
        "AggreFact-CNN": 66.7, "AggreFact-XSum": 76.5,
        "TofuEval-MediaS": 71.4, "TofuEval-MeetB": 79.9,
        "Wice": 80.4, "Reveal": 87.8, "ClaimVerify": 67.6,
        "FactCheck-GPT": 79.9, "ExpertQA": 59.2, "Lfqa": 83.1,
    },
}

comparison_data = {}
for name, scores in baselines.items():
    row = {ds: scores.get(ds, None) for ds in ALL_DATASETS}
    row["Avg"] = round(sum(v for v in scores.values()) / len(scores), 1)
    comparison_data[name] = row

vera_row = {}
for ds in ALL_DATASETS:
    if ds in all_results:
        vera_row[ds] = all_results[ds]["BAcc"]
vera_row["Avg"] = all_results.get("OVERALL", {}).get("BAcc", None)
comparison_data[f"Vera ModernBERT (395M)"] = vera_row

comp_df = pd.DataFrame(comparison_data).T
print("Balanced Accuracy (%) per dataset — comparison with published baselines")
print("-" * 80)
comp_df

Balanced Accuracy (%) per dataset — comparison with published baselines
--------------------------------------------------------------------------------


,AggreFact-CNN,AggreFact-XSum,TofuEval-MediaS,TofuEval-MeetB,Wice,Reveal,ClaimVerify,FactCheck-GPT,ExpertQA,Lfqa,RAGTruth,Avg
AlignScore (355M),52.4,71.4,69.2,72.6,66.0,85.3,69.6,74.3,58.3,84.5,NaN,70.4
MiniCheck-FT5 (770M),69.9,74.3,73.6,77.3,72.2,86.2,74.6,74.7,59.0,85.2,NaN,74.7
GPT-4 (~1.8T),66.7,76.5,71.4,79.9,80.4,87.8,67.6,79.9,59.2,83.1,NaN,75.2
Vera ModernBERT (395M),55.1,52.9,48.7,50.2,48.3,63.5,50.3,57.6,51.5,56.8,54.4,50.1


## 8. Save Results to Drive

In [10]:
os.makedirs(RESULTS_DIR, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

metrics_file = os.path.join(RESULTS_DIR, f"metrics_{SPLIT}_{timestamp}.json")
with open(metrics_file, "w") as f:
    json.dump({
        "model": MODEL_ID,
        "split": SPLIT,
        "threshold": THRESHOLD,
        "timestamp": timestamp,
        "datasets": TARGET_DATASETS,
        "per_dataset": all_results,
    }, f, indent=2)
print(f"✅ Metrics saved to: {metrics_file}")

preds_file = os.path.join(RESULTS_DIR, f"predictions_{SPLIT}_{timestamp}.jsonl")
with open(preds_file, "w") as f:
    for pred in all_predictions:
        f.write(json.dumps(pred) + "\n")
print(f"✅ Predictions saved to: {preds_file}")

csv_file = os.path.join(RESULTS_DIR, f"results_{SPLIT}_{timestamp}.csv")
results_df.to_csv(csv_file)
print(f"✅ Results CSV saved to: {csv_file}")

comp_csv = os.path.join(RESULTS_DIR, f"comparison_{SPLIT}_{timestamp}.csv")
comp_df.to_csv(comp_csv)
print(f"✅ Comparison CSV saved to: {comp_csv}")

✅ Metrics saved to: drive/MyDrive/vera/evaluations/llmaggrefact-modernbert/metrics_test_20260222_215156.json
✅ Predictions saved to: drive/MyDrive/vera/evaluations/llmaggrefact-modernbert/predictions_test_20260222_215156.jsonl
✅ Results CSV saved to: drive/MyDrive/vera/evaluations/llmaggrefact-modernbert/results_test_20260222_215156.csv
✅ Comparison CSV saved to: drive/MyDrive/vera/evaluations/llmaggrefact-modernbert/comparison_test_20260222_215156.csv


In [11]:
from google.colab import runtime
runtime.unassign()